# 09 · Análisis exploratorio de datos (EDA) y estadística

**Duración estimada:** 1h 30min

Vamos a trabajar con el dataset **tips** de seaborn: 244 cuentas de un restaurante con lo que se pagó de cuenta (`total_bill`), la propina (`tip`), y datos del cliente (sexo, si fuma, día, momento del día, tamaño del grupo).

**Lo que vamos a ver:**

1. Primer vistazo con Pandas
2. Medidas de centralidad
3. Medidas de dispersión
4. Representación gráfica
5. Outliers

---

## Datos del caso

Cargamos el dataset `tips` de seaborn. Si no tienes seaborn instalado: `pip install seaborn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

tips = sns.load_dataset('tips')
tips.head()

---

## 1 · Primer vistazo con Pandas

Tres preguntas que hacerte siempre al recibir un dataset:

1. **¿Qué forma tiene?** (filas × columnas)
2. **¿Qué tipos de variables hay?** (números, texto, fechas...)
3. **¿Hay datos faltantes o raros?**

### 1.1 · Información básica

In [ ]:
# Tamaño
tips.shape

In [ ]:
# Nombres de columnas
tips.columns

In [ ]:
# Índice
tips.index

In [ ]:
# Tipos de cada columna
tips.dtypes

**`info()`** resume todo lo anterior en una sola tabla — es lo primero que miro al recibir datos.

In [ ]:
tips.info()

**`describe()`** da estadísticas rápidas solo de las columnas numéricas.

In [ ]:
tips.describe()

In [ ]:
# Para que también incluya las categóricas:
tips.describe(include='all')

### 1.2 · Tipos de variables

Antes de analizar los datos, conviene **clasificar** las variables. Hay dos clasificaciones útiles:

| | Cuantitativas (números) | Cualitativas (categorías) |
|---|---|---|
| **Qué son** | Magnitudes que se pueden sumar | Etiquetas o grupos |
| **Ejemplo** | `total_bill`, `tip`, `size` | `sex`, `smoker`, `day`, `time` |
| **Operaciones** | media, suma, std, percentiles | conteo, moda, tablas cruzadas |

Y dentro de las cuantitativas:

- **Discretas:** valores enteros (nº de personas en la mesa: 1, 2, 3, 4...).
- **Continuas:** cualquier valor en un rango (total_bill: 13.45, 27.83...).

Y dentro de las cualitativas:

- **Nominales:** sin orden (sexo, fumador).
- **Ordinales:** con orden (días de la semana, niveles de satisfacción).

In [ ]:
# Cuantitativas vs cualitativas en tips
cuantitativas = tips.select_dtypes(include=['int64', 'float64']).columns.tolist()
cualitativas  = tips.select_dtypes(include=['object', 'category']).columns.tolist()

print('Cuantitativas:', cuantitativas)
print('Cualitativas :', cualitativas)

### 1.3 · Información básica mediante librerías
Existen librerías que generan **informes automáticos** de un dataset. La más popular hoy es **ydata-profiling** (antes `pandas-profiling`).

In [ ]:
# Instalación (descomentar si hace falta)
# !pip install ydata-profiling -q

In [ ]:
#from ydata_profiling import ProfileReport
#
#report = ProfileReport(tips, title='Tips EDA', minimal=True)
#report.to_notebook_iframe()

> **Nota:** `ydata-profiling` genera un informe muy completo (para cada variable: histogramas, outliers, missings, correlaciones...). Útil para un primer vistazo, pero puede tardar unos segundos en datasets grandes. Para esta sesión hacemos el EDA a mano, que es lo que te da intuición.

### Ejercicios 1 · Primer vistazo

1. ¿Cuántas filas y columnas tiene `tips`?
2. ¿Qué tipo de dato tiene la columna `day`? ¿Es cuantitativa o cualitativa?
3. ¿Hay valores nulos en alguna columna? (Pista: `tips.isna().sum()`.)

In [ ]:
# Tu código aquí

---

## 2 · Medidas de centralidad

Las medidas de centralidad resumen **un conjunto de números en uno solo que representa el "centro"**.

- **Media (promedio):** `tips['tip'].mean()`
- **Mediana (valor que parte los datos por la mitad):** `tips['tip'].median()`
- **Moda (valor que más se repite):** `tips['tip'].mode()`

In [ ]:
tips['tip'].mean()

In [ ]:
tips['tip'].median()

In [ ]:
tips['tip'].mode()

### ¿Cuándo usar cada una?

- **Media** →好用 cuando los datos son "simétricos" y no hay outliers.
- **Mediana** → robusta frente a outliers. Si hay una propina de 1000€, la media se va pero la mediana no.
- **Moda** → para datos cualitativos (día más frecuente) o discretos con pocos valores.

In [ ]:
# ¿Cuál es el día con más cuentas?
tips['day'].mode()

### Centralidad por grupo

In [ ]:
# Propina media por día
tips.groupby('day')['tip'].mean().sort_values(ascending=False)

In [ ]:
# Mediana por día (más robusta si hay outliers)
tips.groupby('day')['tip'].median().sort_values(ascending=False)

### Ejercicios 2 · Centralidad

1. ¿Cuál es la `total_bill` media? ¿Y la mediana? Si difieren, ¿por qué?
2. ¿Cuál es el tamaño de grupo (`size`) más frecuente?
3. ¿Los fumadores dejan más propina en media? (Compara la media de `tip` entre `smoker == 'Yes'` y `smoker == 'No'`.)

In [ ]:
# Tu código aquí

---

## 3 · Medidas de dispersión

Dos datasets pueden tener la **misma media** y ser muy distintos. Las medidas de dispersión miden **cómo de esparcidos** están los datos respecto al centro.

- **Rango** = max − min
- **IQR** = Q3 − Q1 (rango intercuartílico, robusto)
- **Varianza** = media de (x − media)²
- **Desviación típica (std)** = √varianza, en las unidades de los datos
- **Coeficiente de variación** = std / media (adimensional, útil para comparar)

In [ ]:
# Para 'tip'
print('Rango  :', tips['tip'].max() - tips['tip'].min())
print('IQR    :', tips['tip'].quantile(0.75) - tips['tip'].quantile(0.25))
print('Varianza:', tips['tip'].var())
print('Std    :', tips['tip'].std())

In [ ]:
# Coeficiente de variación: ¿qué variable es más dispersa relativamente?
for col in ['total_bill', 'tip', 'size']:
    cv = tips[col].std() / tips[col].mean()
    print(f'{col:12s} → CV = {cv:.2f}')

### Dispersión por grupo

In [ ]:
# Desviación típica de la propina por día
tips.groupby('day')['tip'].std()

### Ejercicios 3 · Dispersión

1. ¿Qué variable es más dispersa en términos absolutos: `total_bill` o `tip`?
2. ¿Y en términos relativos (CV)?
3. ¿Qué día tiene la **propina más variable** (mayor std)? ¿Y la más estable?

In [ ]:
# Tu código aquí

---

## 4 · Representación gráfica

**Regla rápida:**

- **Histograma** → cómo se distribuye UNA variable numérica.
- **Diagrama de barras** → comparar conteos o聚合 entre categorías.
- **Diagrama de líneas** → una variable numérica a lo largo de un **eje ordenado** (tiempo,通常是).

### 4.1 · Histogramas y diagramas de barras

**Histograma** de la propina:

In [ ]:
tips['tip'].hist(bins=20, edgecolor='k')
plt.xlabel('Propina ($)')
plt.ylabel('Frecuencia')
plt.title('Distribución de la propina')
plt.show()

**Diagrama de barras** de cuántos pedidos hubo cada día:

In [ ]:
tips['day'].value_counts().plot.bar(edgecolor='k')
plt.xlabel('Día')
plt.ylabel('Nº de cuentas')
plt.title('Cuentas por día')
plt.xticks(rotation=0)
plt.show()

**Barras de la media** de propina por día:

In [ ]:
tips.groupby('day')['tip'].mean().plot.bar(edgecolor='k')
plt.ylabel('Propina media ($)')
plt.title('Propina media por día')
plt.xticks(rotation=0)
plt.show()

### 4.2 · Diagramas de líneas
Un diagrama de líneas tiene sentido cuando el eje X es **ordenado o continuo** (el caso típico: tiempo).

In [ ]:
# Ejemplo: ventas mensuales de una tienda (datos inventados)
ventas = pd.Series(
    [120, 135, 128, 142, 155, 148, 160, 172, 168, 180, 175, 190],
    index=pd.date_range('2025-01', periods=12, freq='ME'),
    name='ventas'
)
ventas.plot(marker='o')
plt.ylabel('Ventas (miles €)')
plt.title('Ventas mensuales 2025')
plt.show()

**Truco para `tips`:** si ordenas por `total_bill` y dibujas `tip` como línea, ves la **tendencia** de la propina según crece la cuenta (aunque un scatter sería más correcto para dos variables continuas).

In [ ]:
# Tendencia: propina según cuenta (ordenada)
ordenado = tips.sort_values('total_bill')
plt.plot(ordenado['total_bill'], ordenado['tip'], alpha=0.5)
plt.xlabel('Total de la cuenta ($)')
plt.ylabel('Propina ($)')
plt.title('Propina vs cuenta (línea, solo para ver tendencia)')
plt.show()

### Ejercicios 4 · Gráficos

1. Haz un histograma de `total_bill` con `bins=30`. ¿Qué forma tiene?
2. Haz un diagrama de barras de la **mediana** de `tip` por `time` (Lunch / Dinner).
3. ¿En qué se diferencia un histograma de un diagrama de barras?

In [ ]:
# Tu código aquí

---

## 5 · Outliers

**Outlier** = un valor muy alejado del comportamiento habitual del resto.

**¿De dónde vienen?**

- **Error de medida** (el cajero tecleó un 0 de más).
- **Caso raro pero real** (una mesa de 20 personas, una propina de 200€).
- **Mezcla de poblaciones** (el sábado hay un grupo de turistas con propinas muy distintas a las habituales).

**No siempre hay que quitarlos.** A veces son los casos más interesantes.

### 5.1 · Introducción

Veamos el rango de `tip`:

In [ ]:
print('min  :', tips['tip'].min())
print('max  :', tips['tip'].max())
print('media:', round(tips['tip'].mean(), 2))
print('mediana:', tips['tip'].median())

¿Por qué la media (3.00) está más cerca de la mediana (2.90) pero no son iguales? Porque hay **algunas propinas muy altas** que tiran de la media hacia arriba. Veámoslo en un histograma:

### 5.2 · Histogramas

In [ ]:
tips['tip'].hist(bins=30, edgecolor='k')
plt.xlabel('Propina ($)')
plt.ylabel('Frecuencia')
plt.title('Distribución de propinas')
plt.axvline(tips['tip'].mean(), color='red', linestyle='--', label='media')
plt.legend()
plt.show()

¿Ves la **larga cola a la derecha**? Esos valores entre 6€ y 11€ son los candidatos a outliers.

### 5.3 · Boxplots
El boxplot es la forma compacta de ver la distribución **y los outliers** en una sola imagen.

In [ ]:
tips['tip'].plot.box()
plt.ylabel('Propina ($)')
plt.title('Boxplot de la propina')
plt.show()

**Cómo se lee un boxplot:**

```
        ┌──┐  ← bigote superior (Q3 + 1.5·IQR)
        │  │
     ┌──┤  ├──┐
     │  │  │  │
     │  ├──┤  │  ← caja: del Q1 al Q3 (el 50% central)
     │  │  │  │
     └──┤  ├──┘
        │  │
        └──┘  ← bigote inferior (Q1 - 1.5·IQR)
          •   ← outlier (fuera de los bigotes)


In [ ]:
# Boxplot por día (útil para comparar distribuciones)
tips.boxplot(column='tip', by='day')
plt.suptitle('')  # quita el título automático
plt.title('Propina por día')
plt.ylabel('Propina ($)')
plt.show()

### 5.4 · Métodos numéricos
Para **detectar** outliers de forma automática, hay dos métodos clásicos:

**Regla del IQR (la que usa el boxplot por defecto):**

Un valor es outlier si queda fuera de `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`.

In [ ]:
q1 = tips['tip'].quantile(0.25)
q3 = tips['tip'].quantile(0.75)
iqr = q3 - q1
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr:.2f}')
print(f'Límites: [{lim_inf:.2f}, {lim_sup:.2f}]')

In [ ]:
mask_outlier = (tips['tip'] < lim_inf) | (tips['tip'] > lim_sup)
print(f'Outliers detectados: {mask_outlier.sum()}')
tips[mask_outlier].head()

**Z-score (basado en la media y std):**

Un valor es outlier si su z-score es > 3 (o < -3) en valor absoluto.

`z = (x − media) / std`

In [ ]:
from scipy import stats

z = stats.zscore(tips['tip'])
mask_z = np.abs(z) > 2  # umbral laxo para ver varios
print(f'Outliers (|z| > 2): {mask_z.sum()}')
tips[mask_z].head()

**¿Cuándo usar cada uno?**

| Método | Cuándo |
|---|---|
| **IQR** | Cuando hay outliers que tiran de la media (es robusto). El estándar. |
| **Z-score** | Cuando los datos son aproximadamente normales. Más sensible. |

### 5.5 · Outliers en varias dimensiones
Hasta ahora hemos buscado outliers **en una sola variable**. Pero un punto puede ser perfectamente normal en X y en Y... y ser raro **en la combinación**.

In [ ]:
# Scatter básico: cuenta vs propina
tips.plot.scatter(x='total_bill', y='tip', alpha=0.6)
plt.xlabel('Total de la cuenta ($)')
plt.ylabel('Propina ($)')
plt.title('Cuenta vs propina')
plt.show()

**¿Ves los puntos arriba a la derecha?** Son cuentas grandes con propina grande — **no son outliers** porque siguen la tendencia.

**¿Y el de abajo a la izquierda con cuenta grande y propina casi cero?** Ese sí es un outlier multivariante: su `total_bill` es normal pero su `tip` es anormalmente bajo para esa cuenta.

In [ ]:
# Resaltamos los outliers univariantes de 'tip' (los que detectamos con IQR)
mask_outlier = (tips['tip'] < lim_inf) | (tips['tip'] > lim_sup)

ax = tips.plot.scatter(x='total_bill', y='tip', alpha=0.4, label='normal')
tips[mask_outlier].plot.scatter(x='total_bill', y='tip',
                                color='red', s=80, label='outlier univariante', ax=ax)
plt.title('Outliers de tip (univariante) sobre el scatter')
plt.legend()
plt.show()

**Truco multivariante simple:** un punto es candidato a outlier multivariante si es outlier en **al menos una** de las variables, **o** si está muy lejos de la "nube" principal visualmente. Para algo más serio se usa la **distancia de Mahalanobis**, que tiene en cuenta la correlación entre variables.

In [ ]:
# Outliers multivariantes: flag si es outlier en 'tip' O en 'total_bill'
q1b, q3b = tips['total_bill'].quantile([0.25, 0.75])
iqr_b = q3b - q1b
mask_b = (tips['total_bill'] < q1b - 1.5*iqr_b) | (tips['total_bill'] > q3b + 1.5*iqr_b)

mask_multi = mask_outlier | mask_b
print(f'Outliers multivariantes (IQR en tip o total_bill): {mask_multi.sum()}')

ax = tips.plot.scatter(x='total_bill', y='tip', alpha=0.3, label='normal')
tips[mask_multi].plot.scatter(x='total_bill', y='tip',
                              color='red', s=80, label='outlier multivariante', ax=ax)
plt.title('Outliers multivariantes')
plt.legend()
plt.show()

### Ejercicios 5 · Outliers

1. Para `total_bill`, calcula Q1, Q3 e IQR. ¿Cuántos outliers detecta la regla del IQR?
2. Haz un boxplot de `total_bill` agrupado por `time` (Lunch vs Dinner). ¿En cuál hay más outliers?
3. ¿Cuántos outliers detecta la regla del IQR en `tip` si el umbral es 3·IQR en vez de 1.5·IQR? (Pista: solo cambia el 1.5 por 3.)

In [ ]:
# Tu código aquí

---

## Cierre

| Pregunta | Herramienta |
|---|---|
| ¿Qué forma tienen los datos? | `info`, `describe`, histogramas |
| ¿Cuál es el valor "típico"? | media, mediana, moda |
| ¿Qué tan esparcidos están? | rango, IQR, std, CV |
| ¿Cómo se distribuyen? | histograma, barras, líneas |
| ¿Hay valores raros? | boxplot, IQR, z-score, scatter multivariante |

**Para qué sirve esto en la realidad:**

- Antes de cualquier modelo: **entender los datos** (outliers, distribuciones, missing).
- En un informe: **comunicar** con gráficos claros.
- En una decisión: **no fiarte solo de la media**, mira la dispersión y los outliers.